## Final model - CE, up to 40 epochs

In [ ]:
BRANCH = "main"

%cd /tmp
!rm -rf damage-detection
!git clone -b {BRANCH} https://github.com/azrairemd/damage-detection.git
%cd damage-detection
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
import os
import shutil
import time
import torch
from torch.utils.data import DataLoader, Subset

from src.data.dataset import XBDDataset
from src.losses.combined_loss import CombinedLoss
from src.models.siamese_unet import SiameseUNet
from src.train import fit, train_one_epoch, validate
from src.utils.config import load_config

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
CONFIG_PATH = "configs/siamese_ce_long.yaml"

In [ ]:
config = load_config(CONFIG_PATH)
data = config["data"]
train_cfg = config["training"]
loss_cfg = config["loss"]
model_cfg = config["model"]

In [ ]:
# data
train_ds = XBDDataset(data["train_split"], data["xbd_root"], data["mask_root"],
                      train=True, crop_size=data["crop_size"])
val_ds = XBDDataset(data["val_split"], data["xbd_root"], data["mask_root"], train=False)

train_loader = DataLoader(train_ds, batch_size=train_cfg["batch_size"], shuffle=True,
                          num_workers=train_cfg["num_workers"], drop_last=True)
val_loader = DataLoader(val_ds, batch_size=train_cfg["val_batch_size"], shuffle=False,
                        num_workers=train_cfg["num_workers"])

# model, loss and optimizer
model = SiameseUNet(encoder=model_cfg["encoder"], fusion=model_cfg["fusion"]).to(device)
loss_fn = CombinedLoss(alpha=torch.tensor(loss_cfg["class_weights"]),
                       gamma=loss_cfg["gamma"], lam=loss_cfg["lam"])
optimizer = torch.optim.AdamW(model.parameters(), lr=train_cfg["learning_rate"])

In [ ]:
out_dir = train_cfg["out_dir"]
os.makedirs(out_dir, exist_ok=True)
shutil.copy(CONFIG_PATH, os.path.join(out_dir, "config.yaml"))

# copy the checkpoint of the 20-epoch CE run, to continue from epoch 21
CHECKPOINT_DIR = "/kaggle/input/datasets/azra41221/siamese-ce-checkpoint"
for name in ["last.pt", "best.pt", "log.csv"]:
    shutil.copy(os.path.join(CHECKPOINT_DIR, name), os.path.join(out_dir, name))

start = time.time()
best = fit(model, train_loader, val_loader, loss_fn, optimizer, device,
           max_epochs=train_cfg["max_epochs"], out_dir=out_dir, resume=True)

print(f"Best val score: {best:.4f}")
print(f"Total time: {(time.time() - start) / 3600:.1f} hours")

**Final model results (Siamese concat + class-weighted CE, continued to 40 epochs)**

- Training was resumed from the 20-epoch CE checkpoint (`last.pt`) with `max_epochs = 40` (patience 8).
- **Best validation xView2 score: 0.7401 at epoch 35** (localization F1 0.793, damage F1 0.718). New best scores at epochs 22, 30, 33, 34 and 35; no improvement in epochs 36–40. Mean of the 3 best epochs: 0.738.
- Extending training improved the score by +0.032 (0.7079 → 0.7401), mostly through minor damage (0.463 → 0.547).
- Compared to the baseline (0.6095), the final model is better for every class: no damage 0.926 (+0.10), minor 0.547 (+0.18), major 0.675 (+0.06), destroyed 0.846 (+0.20).